In [1]:
from pyspark.sql import functions as F

# ---------------------------------
# LOAD OUTPUT TABLES FROM NOTEBOOK 2
# ---------------------------------
enriched = spark.table("po_enriched_lines")
validation_results = spark.table("po_validation_results")

# ---------------------------------
# ORDER-LEVEL DECISION
# ---------------------------------
order_decision = (
    enriched
    .groupBy(
        "document_id",
        "customer_id",
        "sales_order_no",
        "source_file_name",
        "sales_origin"
    )
    .agg(
        F.sum("blocking_error_count").alias("blocking_error_count"),
        F.sum(
            F.when(F.col("final_line_status") == "REQUIRES_REVIEW", 1).otherwise(0)
        ).alias("review_line_count"),
        F.first("doc_total").alias("document_total"),
        F.first("calculated_doc_total").alias("calculated_total")
    )
    .withColumn(
        "final_order_status",
        F.when(F.col("blocking_error_count") > 0, F.lit("BLOCKED"))
         .when(F.col("review_line_count") > 0, F.lit("REQUIRES_REVIEW"))
         .otherwise(F.lit("READY_TO_SAVE"))
    )
)

# ---------------------------------
# SAVE ORDER DECISION TABLE
# ---------------------------------
(
    order_decision.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("po_order_decision")
)

# ---------------------------------
# SPLIT INTO READY / REVIEW / BLOCKED
# ---------------------------------
ready_orders = order_decision.filter(F.col("final_order_status") == "READY_TO_SAVE")
review_orders = order_decision.filter(F.col("final_order_status") == "REQUIRES_REVIEW")
blocked_orders = order_decision.filter(F.col("final_order_status") == "BLOCKED")

(
    ready_orders.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("po_ready_orders")
)

(
    review_orders.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("po_review_orders")
)

(
    blocked_orders.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("po_blocked_orders")
)

# ---------------------------------
# EXCEPTION DETAIL
# ---------------------------------
exception_detail = (
    enriched
    .filter(F.col("final_line_status") != "READY")
    .select(
        "document_id",
        "line_no",
        "item_extracted",
        "description_extracted",
        "matched_item_no",
        "matched_item_desc",
        "match_type",
        "order_status_validation_status",
        "unit_price_validation_status",
        "total_validation_status",
        "uom_validation_status",
        "moq_validation_status",
        "item_hold_validation_status",
        "final_line_status",
        "sales_origin"
    )
)

(
    exception_detail.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("po_exception_detail")
)

# ---------------------------------
# DEMO STORY TABLE
# ---------------------------------
demo_story = spark.createDataFrame([
    ("DOC001", "Valid happy path", "READY_TO_SAVE"),
    ("DOC002", "Sales order not in Journal", "BLOCKED"),
    ("DOC003", "Unit price missing", "REQUIRES_REVIEW"),
    ("DOC004", "Header total mismatch", "BLOCKED"),
    ("DOC005", "Close item match suggestion", "REQUIRES_REVIEW"),
    ("DOC006", "No item match found", "BLOCKED"),
    ("DOC007", "UOM conversion successful", "READY_TO_SAVE"),
    ("DOC008", "Missing UOM conversion", "BLOCKED"),
    ("DOC009", "Minimum order quantity failure", "BLOCKED"),
    ("DOC010", "Item on hold", "BLOCKED"),
], ["document_id", "scenario", "expected_status"])

# ---------------------------------
# SUMMARY VIEW
# ---------------------------------
summary = (
    order_decision
    .groupBy("final_order_status")
    .count()
    .orderBy("final_order_status")
)

# ---------------------------------
# DISPLAY RESULTS
# ---------------------------------
print("Order Decision Summary")
display(summary)

print("All Orders")
display(order_decision.orderBy("document_id"))

print("Ready Orders")
display(ready_orders.orderBy("document_id"))

print("Review Orders")
display(review_orders.orderBy("document_id"))

print("Blocked Orders")
display(blocked_orders.orderBy("document_id"))

print("Exception Detail")
display(exception_detail.orderBy("document_id", "line_no"))

print("Validation Results")
display(validation_results.orderBy("document_id", "line_no", "rule_name"))

print("Expected Demo Story")
display(demo_story.orderBy("document_id"))

StatementMeta(, cb2d0dd0-d204-4288-8d1a-e9f257201f21, 3, Finished, Available, Finished, False)

Order Decision Summary


SynapseWidget(Synapse.DataFrame, 983814eb-0dc6-4a2c-a7b6-3dcdea648614)

All Orders


SynapseWidget(Synapse.DataFrame, 8084644c-80dc-4b2b-8861-e7d5ba83d039)

...

SynapseWidget(Synapse.DataFrame, 6f649c58-fc3c-4348-86b8-a8b78b6e1b3c)

Validation Results


SynapseWidget(Synapse.DataFrame, 9beae82c-8f5d-4e5b-bc8c-475fac13d7ca)

Expected Demo Story


SynapseWidget(Synapse.DataFrame, 74b91f52-55c9-455d-ad32-e0852c90aac4)